# Re-Verification — Reproductive Entities (OpenAI Batch API)

Runs `prompt_verification_reprod_new.md` on every JSON file in `dataset_gpt_reverif/`  
using the **OpenAI Batch API** (async, 50 % cost discount, 24 h window) with **gpt-4.1**.

Requests are submitted in **batches of 100** (`BATCH_SIZE`). If more than 100 files are
pending, several batch jobs are created and tracked together in `_batch_ids_gpt41.json`.

### Workflow
| Cell | Action |
|------|--------|
| **1 — Config** | Set paths, model, batch size, temperature |
| **2 — Submit** | Build JSONL per chunk of 100, upload, create batch job(s) |
| **3 — Poll**   | Check every batch's status until all complete |
| **4 — Apply**  | Download results, write `verification_reprod` into each JSON file |
| **5 — Summary**| PASS / FAIL / GENERATION_FAILURE counts |
| **6 — FAIL detail** | List files with critical errors |
| **7 — GENERATION_FAILURE detail** | List generation failures |

**Run cells 2 -> 3 -> 4 in order. Cell 3 can be re-run until all batches = `completed`.**

> **Model note:** `gpt-4.1` supports the Chat-Completions Batch API with the standard
> `temperature` and `max_tokens` parameters (no `reasoning_effort`). Use `gpt-4.1-mini`
> for a cheaper/faster variant or `gpt-4.1-nano` for the lightest option — all three
> work identically with this notebook by changing `MODEL`.

In [1]:
import json, os, time, urllib.request, urllib.error, io
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# find_dotenv() walks up from cwd until it finds .env — works in Jupyter regardless of __file__
load_dotenv(find_dotenv())

# -- Configuration ------------------------------------------------------------
API_KEY    = os.environ.get("OPENAI_API_KEY")

MODEL      = "gpt-4.1"          # or "gpt-4.1-mini" / "gpt-4.1-nano"

INPUT_DIR   = Path("../dataset_gpt_4_reverif")
PROMPT_FILE = Path("../verif_prompts/prompt_verification_reprod_new.md")

BATCH_IDS_FILE = Path("../dataset_gpt_reverif/_batch_ids_gpt41.json")

# -- Batch / generation parameters --------------------------------------------
BATCH_SIZE        = 100
TEMPERATURE       = 0.2
MAX_TOKENS        = 16000
SEED              = 7

OAI_BASE   = "https://api.openai.com/v1"

# -----------------------------------------------------------------------------
if not API_KEY:
    raise ValueError("OPENAI_API_KEY not set -- check your .env file.")

SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")
json_files    = sorted(INPUT_DIR.glob("*.json"))
json_files    = [f for f in json_files if not f.name.startswith("_")]

already_done  = [f for f in json_files
                 if json.loads(f.read_text()).get("verification_reprod") is not None]
pending       = [f for f in json_files
                 if json.loads(f.read_text()).get("verification_reprod") is None
                 and json.loads(f.read_text()).get("corrected_text")]

chunks = [pending[i:i + BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]

print(f"Model             : {MODEL}")
print(f"Temperature       : {TEMPERATURE}")
print(f"Max tokens        : {MAX_TOKENS}")
print(f"Seed              : {SEED}")
print(f"Prompt file       : {PROMPT_FILE.name} ({len(SYSTEM_PROMPT):,} chars)")
print(f"Total files       : {len(json_files)}")
print(f"Already verified  : {len(already_done)}")
print(f"Pending           : {len(pending)}")
print(f"Batch size        : {BATCH_SIZE}")
print(f"=> will submit {len(chunks)} batch job(s)")

Model             : gpt-4.1
Temperature       : 0.2
Max tokens        : 16000
Seed              : 7
Prompt file       : prompt_verification_reprod_new.md (27,749 chars)
Total files       : 129
Already verified  : 0
Pending           : 129
Batch size        : 100
=> will submit 2 batch job(s)


## Cell 2 — Submit Batches
Builds one JSONL request file per chunk of `BATCH_SIZE` (100) pending files, uploads each
to the OpenAI Files API, and creates a batch job for each.  
**Only run this once.** All batch IDs are saved to `_batch_ids.json` so you can resume.
Delete that file to start a fresh run.

In [2]:
def oai_request(method: str, path: str, data=None, headers_extra: dict = None,
                raw_body: bytes = None, content_type: str = "application/json") -> dict:
    """Thin wrapper around urllib for OpenAI REST calls."""
    url = f"{OAI_BASE}{path}"
    headers = {"Authorization": f"Bearer {API_KEY}"}
    if headers_extra:
        headers.update(headers_extra)

    if raw_body is not None:
        body = raw_body
    elif data is not None:
        body = json.dumps(data).encode()
        headers["Content-Type"] = content_type
    else:
        body = None

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())


def oai_download(path: str) -> bytes:
    """Download raw bytes (for batch output file)."""
    url = f"{OAI_BASE}{path}"
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {API_KEY}"}, method="GET")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return resp.read()


def upload_multipart_jsonl(jsonl_bytes: bytes, filename: str) -> str:
    """Upload a JSONL file via multipart/form-data and return the file_id."""
    boundary = b"----OAIBoundary7f3k9x"
    body = (
        b"--" + boundary + b"\r\n"
        b'Content-Disposition: form-data; name="purpose"\r\n\r\n'
        b"batch\r\n"
        b"--" + boundary + b"\r\n"
        + f'Content-Disposition: form-data; name="file"; filename="{filename}"\r\n'.encode()
        + b"Content-Type: application/jsonl\r\n\r\n"
        + jsonl_bytes + b"\r\n"
        b"--" + boundary + b"--\r\n"
    )
    ct = f"multipart/form-data; boundary={boundary.decode()}"
    url = f"{OAI_BASE}/files"
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": ct}
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())["id"]


def build_request_obj(fp):
    """One JSONL line: a Chat-Completions request for gpt-4.1."""
    data = json.loads(fp.read_text(encoding="utf-8"))
    user_payload = json.dumps({
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "specie_en":      data.get("specie_en"),
        "corrected_text": data.get("corrected_text"),
    }, ensure_ascii=False)

    return {
        "custom_id": fp.stem,           # filename without .json -- matches results back
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model":       MODEL,
            "temperature": TEMPERATURE,
            "max_tokens":  MAX_TOKENS,
            "seed":        SEED,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_payload},
            ],
        },
    }


# Guard: don't re-submit if batches are already running
if BATCH_IDS_FILE.exists():
    existing = json.loads(BATCH_IDS_FILE.read_text())
    print(f"Batches already submitted: {len(existing)} job(s)")
    for b in existing:
        print(f"  {b['batch_id']}  ({b['n']} requests)")
    print("Skip to Cell 3 (Poll) / Cell 4 (Apply).")
    print("Delete _batch_ids_gpt41.json to start a fresh run.")
else:
    if not pending:
        print("Nothing to submit -- all files already verified.")
    else:
        submitted = []
        for idx, chunk in enumerate(chunks, 1):
            lines_jsonl = [json.dumps(build_request_obj(fp), ensure_ascii=False) for fp in chunk]
            jsonl_bytes = "\n".join(lines_jsonl).encode("utf-8")
            print(f"[chunk {idx}/{len(chunks)}] {len(lines_jsonl)} requests, {len(jsonl_bytes):,} bytes")

            file_id = upload_multipart_jsonl(jsonl_bytes, f"reverif_batch_{idx}.jsonl")
            print(f"  uploaded -> {file_id}")

            batch = oai_request("POST", "/batches", data={
                "input_file_id":     file_id,
                "endpoint":          "/v1/chat/completions",
                "completion_window": "24h",
                "metadata": {"description": f"DEB reverif reprod -- {MODEL} -- chunk {idx}"},
            })
            submitted.append({"batch_id": batch["id"], "file_id": file_id,
                              "n": len(lines_jsonl), "status": batch["status"]})
            print(f"  created -> {batch['id']}  (status: {batch['status']})")

        BATCH_IDS_FILE.write_text(json.dumps(submitted, indent=2))
        print(f"\nSubmitted {len(submitted)} batch job(s). IDs saved to {BATCH_IDS_FILE}")
        print("Now run Cell 3 to poll.")

[chunk 1/2] 100 requests, 3,502,878 bytes
  uploaded -> file-KzjzeSGgACuaQ27aQsNqcz
  created -> batch_6a4105496f948190a7e8a81a65e7d4b6  (status: validating)
[chunk 2/2] 29 requests, 1,004,232 bytes
  uploaded -> file-5s4MyUzHECVXFRJNrdKFCB
  created -> batch_6a41054bc2e4819087b48f2426045b3d  (status: validating)

Submitted 2 batch job(s). IDs saved to ../dataset_gpt_reverif/_batch_ids_gpt41.json
Now run Cell 3 to poll.


## Cell 3 — Poll Status
Re-run until **every** batch shows `status = completed`.  
OpenAI typically completes batches of ~100 requests in minutes to a few hours.

In [6]:
if not BATCH_IDS_FILE.exists():
    print("No batch IDs found. Run Cell 2 first.")
else:
    batches = json.loads(BATCH_IDS_FILE.read_text())
    all_done = True
    tot_completed = tot_failed = tot_all = 0

    for b in batches:
        info   = oai_request("GET", f"/batches/{b['batch_id']}")
        status = info["status"]
        counts = info.get("request_counts", {})
        tot_completed += counts.get("completed", 0)
        tot_failed    += counts.get("failed", 0)
        tot_all       += counts.get("total", 0)
        print(f"{b['batch_id']}  status={status:<12} "
              f"completed={counts.get('completed','-')}/{counts.get('total','-')} "
              f"failed={counts.get('failed','-')}")
        if status != "completed":
            all_done = False
        if status in ("failed", "expired", "cancelled"):
            errs = info.get("errors")
            if errs:
                print("   ", json.dumps(errs))

    print(f"\nAggregate: {tot_completed}/{tot_all} completed, {tot_failed} failed")
    if all_done:
        print("All batches complete! Run Cell 4 to apply results.")
    else:
        print("Still processing -- re-run this cell in a few minutes.")

batch_6a4105496f948190a7e8a81a65e7d4b6  status=completed    completed=100/100 failed=0
batch_6a41054bc2e4819087b48f2426045b3d  status=completed    completed=29/29 failed=0

Aggregate: 129/129 completed, 0 failed
All batches complete! Run Cell 4 to apply results.


## Cell 4 — Apply Results
Downloads each completed batch's output JSONL and writes `verification_reprod` into the
source JSON files. Safe to re-run: already-applied files are skipped.

In [7]:
if not BATCH_IDS_FILE.exists():
    print("No batch IDs found. Run Cell 2 first.")
else:
    batches = json.loads(BATCH_IDS_FILE.read_text())
    applied = skipped = api_errors = parse_errors = 0

    def apply_line(result):
        global applied, skipped, api_errors, parse_errors
        custom_id = result["custom_id"]
        filepath  = INPUT_DIR / f"{custom_id}.json"
        if not filepath.exists():
            print(f"  [WARN] file not found for '{custom_id}'")
            return
        data = json.loads(filepath.read_text(encoding="utf-8"))
        if data.get("verification_reprod") is not None:
            skipped += 1
            return
        if result.get("error"):
            print(f"  [API-ERROR] {custom_id}: {result['error']}")
            api_errors += 1
            return

        body = result.get("response", {}).get("body", {})
        raw  = body.get("choices", [{}])[0].get("message", {}).get("content", "")
        try:
            verification = json.loads(raw)
        except json.JSONDecodeError:
            stripped = raw.strip().removeprefix("```json").removesuffix("```").strip()
            try:
                verification = json.loads(stripped)
            except json.JSONDecodeError:
                print(f"  [PARSE-ERROR] {custom_id}: could not parse model output")
                parse_errors += 1
                return

        data["verification_reprod"] = verification
        filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        u = body.get("usage", {})
        print(f"  {custom_id} -> {verification.get('verdict','?')} | "
              f"{verification.get('critical_errors','?')} critical | "
              f"{u.get('prompt_tokens','?')} in / {u.get('completion_tokens','?')} out")
        applied += 1

    for b in batches:
        info = oai_request("GET", f"/batches/{b['batch_id']}")
        if info["status"] != "completed":
            print(f"[skip] {b['batch_id']} status={info['status']} (not completed yet)")
            continue
        out_id = info.get("output_file_id")
        print(f"[batch {b['batch_id']}] downloading {out_id} ...")
        raw_output = oai_download(f"/files/{out_id}/content")
        for line in raw_output.decode("utf-8").splitlines():
            if line.strip():
                apply_line(json.loads(line))
        err_id = info.get("error_file_id")
        if err_id:
            print(f"  error file present ({err_id}):")
            raw_err = oai_download(f"/files/{err_id}/content")
            for line in raw_err.decode("utf-8").splitlines():
                if line.strip():
                    print("   ", json.loads(line))

    print(f"\nDone: {applied} applied, {skipped} already done, "
          f"{api_errors} API errors, {parse_errors} parse errors")

[batch batch_6a4105496f948190a7e8a81a65e7d4b6] downloading file-Q9GUU1zTsMxA1gZh2fYk7v ...
  0067_Achatina_achatina_abj -> FAIL | 1 critical | 8127 in / 1386 out
  0079_Acipenser_nudiventris_std -> FAIL | 1 critical | 8082 in / 1306 out
  0248_Alosa_kessleri_abj -> FAIL | 1 critical | 8152 in / 1474 out
  0271_Amazona_amazonica_std -> FAIL | 2 critical | 8056 in / 1486 out
  0295_Ambystoma_jeffersonianum_std -> FAIL | 2 critical | 8101 in / 1512 out
  0301_Ambystoma_tigrinum_std -> FAIL | 1 critical | 7983 in / 1236 out
  0349_Anadia_bogotensis_std -> FAIL | 2 critical | 8105 in / 1619 out
  0391_Angustassiminea_castanea_abj -> PASS | 0 critical | 8102 in / 70 out
  0440_Antigone_rubicunda_std -> FAIL | 2 critical | 8169 in / 1620 out
  0566_Ariopsis_felis_abj -> PASS | 0 critical | 8114 in / 69 out
  0593_Artibeus_watsoni_stx -> FAIL | 2 critical | 8086 in / 1474 out
  0752_Bathyraja_maculata_std -> PASS | 0 critical | 8084 in / 71 out
  0755_Bathyraja_parmifera_std -> FAIL | 2 critic

## Cell 5 — Summary

In [8]:
from collections import Counter

counts = Counter()
for filepath in sorted(INPUT_DIR.glob("*.json")):
    if filepath.name.startswith("_"):
        continue
    data = json.loads(filepath.read_text(encoding="utf-8"))
    if not data.get("corrected_text"):
        counts["NO_TEXT"] += 1
        continue
    v = data.get("verification_reprod")
    if v is None:
        counts["NOT_VERIFIED"] += 1
    else:
        counts[v.get("verdict", "UNKNOWN")] += 1

total = sum(counts.values())
print(f"=== Re-Verification Summary ({total} files) ===")
for verdict, count in counts.most_common():
    print(f"  {verdict}: {count}")

=== Re-Verification Summary (129 files) ===
  FAIL: 102
  PASS: 26
  SPAN-B: 1


## Cell 6 — FAIL detail

In [9]:
fails = []
for filepath in sorted(INPUT_DIR.glob("*.json")):
    if filepath.name.startswith("_"):
        continue
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification_reprod")
    if v and v.get("verdict") == "FAIL":
        fails.append({
            "file":            filepath.name,
            "specie":          data.get("specie"),
            "critical_errors": v.get("critical_errors"),
            "error_codes":     [e.get("code") for e in v.get("errors", [])],
        })

print(f"FAIL ({len(fails)} files):")
for r in fails:
    print(f"  {r['file']}  —  {r['specie']}  —  {r['critical_errors']} critical  —  {r['error_codes']}")

FAIL (102 files):
  0067_Achatina_achatina_abj.json  —  Achatina_achatina  —  1 critical  —  ['SPAN-C']
  0079_Acipenser_nudiventris_std.json  —  Acipenser_nudiventris  —  1 critical  —  ['SPAN-C']
  0248_Alosa_kessleri_abj.json  —  Alosa_kessleri  —  1 critical  —  ['SPAN-A+SPAN-C']
  0271_Amazona_amazonica_std.json  —  Amazona_amazonica  —  2 critical  —  ['SPAN-C', 'SPAN-C']
  0295_Ambystoma_jeffersonianum_std.json  —  Ambystoma_jeffersonianum  —  2 critical  —  ['SPAN-C', 'SPAN-A']
  0301_Ambystoma_tigrinum_std.json  —  Ambystoma_tigrinum  —  1 critical  —  ['SPAN-C']
  0349_Anadia_bogotensis_std.json  —  Anadia_bogotensis  —  2 critical  —  ['SPAN-C', 'SPAN-C', 'SPAN-C']
  0440_Antigone_rubicunda_std.json  —  Antigone_rubicunda  —  2 critical  —  ['G1c', 'SPAN-C']
  0593_Artibeus_watsoni_stx.json  —  Artibeus_watsoni  —  2 critical  —  ['SPAN-C', 'SPAN-A']
  0755_Bathyraja_parmifera_std.json  —  Bathyraja_parmifera  —  2 critical  —  ['SPAN-C', 'SPAN-C']
  0895_Bubalus_bubalis_stx

## Cell 7 — GENERATION_FAILURE detail

In [10]:
failures = []
for filepath in sorted(INPUT_DIR.glob("*.json")):
    if filepath.name.startswith("_"):
        continue
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification_reprod")
    if v and v.get("verdict") == "GENERATION_FAILURE":
        failures.append({"file": filepath.name, "specie": data.get("specie")})

print(f"GENERATION_FAILURE ({len(failures)} files):")
for r in failures:
    print(f"  {r['file']}  —  {r['specie']}")

GENERATION_FAILURE (0 files):
